# Holdout Evaluation / Prediction

Loads the saved ALBERT model and evaluates it on the held-out rows
(`holdout_eval.csv`) that were **not** used during training or testing.

**Before running:** upload `albert_complaint_model.zip` (or the unzipped folder)
and `holdout_eval.csv` produced by the training notebook.

In [ ]:
# ── Step 1: Install packages ──
!pip install -q transformers==4.41.2

In [ ]:
# ── Step 2: Upload model zip + holdout CSV ──
# Upload: albert_complaint_model.zip  AND  holdout_eval.csv
from google.colab import files
uploaded = files.upload()

In [ ]:
# ── Step 3: Unzip the model if needed ──
import os
if os.path.exists('albert_complaint_model.zip') and not os.path.isdir('albert_complaint_model'):
    !unzip -o -q albert_complaint_model.zip
print(os.listdir('.'))

In [ ]:
# ── Step 4: Imports & config ──
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)
from transformers import AlbertTokenizer, AlbertForSequenceClassification

MODEL_DIR   = './albert_complaint_model'
HOLDOUT_CSV = 'holdout_eval.csv'
MAX_LEN     = 128
BATCH_SIZE  = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Step 5: Load model, tokenizer & label mapping ──
tokenizer = AlbertTokenizer.from_pretrained(MODEL_DIR)
model     = AlbertForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

label_map = pd.read_csv(f'{MODEL_DIR}/label_mapping.csv')
id2cat    = dict(zip(label_map['label'], label_map['category']))
cat2id    = {v: k for k, v in id2cat.items()}
class_names = [id2cat[i] for i in range(len(id2cat))]
print(f'Loaded model with {len(class_names)} classes.')

In [ ]:
# ── Step 6: Load holdout data ──
holdout = pd.read_csv(HOLDOUT_CSV)
holdout = holdout[['Issue_Summary', 'Complaint Category Description']].dropna()
holdout['Issue_Summary'] = holdout['Issue_Summary'].astype(str).str.strip()
holdout = holdout[holdout['Issue_Summary'] != ''].reset_index(drop=True)

# Keep only rows whose label the model actually knows about
holdout = holdout[holdout['Complaint Category Description'].isin(cat2id)].reset_index(drop=True)
holdout['label'] = holdout['Complaint Category Description'].map(cat2id)

print(f'Holdout rows for evaluation: {len(holdout):,}')

In [ ]:
# ── Step 7: Dataset & dataloader ──
class ComplaintDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'token_type_ids' : enc['token_type_ids'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long),
        }

eval_ds     = ComplaintDataset(holdout['Issue_Summary'], holdout['label'], tokenizer, MAX_LEN)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Eval batches: {len(eval_loader)}')

In [ ]:
# ── Step 8: Run predictions ──
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in eval_loader:
        out = model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            token_type_ids = batch['token_type_ids'].to(device),
        )
        all_preds.extend(torch.argmax(out.logits, 1).cpu().numpy())
        all_labels.extend(batch['label'].numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
print('Predictions complete.')

In [ ]:
# ── Step 9: Metrics on the holdout set ──
acc = accuracy_score(all_labels, all_preds)
p   = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
r   = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1  = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print('── Holdout Evaluation ──')
print(f'Rows     : {len(all_labels):,}')
print(f'Accuracy : {acc:.4f}')
print(f'Precision: {p:.4f}')
print(f'Recall   : {r:.4f}')
print(f'F1       : {f1:.4f}')
print()

labels_present = sorted(set(all_labels) | set(all_preds))
print(classification_report(
    all_labels, all_preds,
    labels=labels_present,
    target_names=[id2cat[i] for i in labels_present],
    zero_division=0
))

In [ ]:
# ── Step 10: Save per-row predictions ──
holdout_out = holdout.copy()
holdout_out['predicted_label']    = all_preds
holdout_out['predicted_category'] = [id2cat[i] for i in all_preds]
holdout_out['correct']            = holdout_out['label'] == holdout_out['predicted_label']
holdout_out.to_csv('holdout_predictions.csv', index=False)

print('Saved holdout_predictions.csv')
files.download('holdout_predictions.csv')